## 1. Install, imports, upload helper

In [ ]:
# Cell 1: installs, imports, paths, upload helper
import sys, os, json, math, re, shutil, zipfile, subprocess, datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'rasterio', 'geopandas', 'pyproj', 'shapely', 'pillow', 'matplotlib', 'requests'], check=False)

import numpy as np
import pandas as pd

# Folders (IN = everything you upload, SITE/data = output contract)
IN   = Path('/content/in')   if IN_COLAB else Path(os.environ.get('IN_DIR', 'in'))
SITE = Path('/content/site') if IN_COLAB else Path(os.environ.get('SITE_DIR', 'site'))
OUT  = SITE / 'data'
IN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

# Files that must be in IN (3 GeoTIFFs are checked separately in cell 2)
NEEDED_DATA = ['best_compromise_10x.csv', 'pareto_solutions_10x.csv', 'site_turbine_table.csv',
               'candidate_sites.csv', 'accessibility_points.csv', 'custom.geojson']   # region_config.json is not needed here

def upload_here():
    """Colab upload dialog; files are saved into IN."""
    from google.colab import files
    for name, data in files.upload().items():
        (IN / name).write_bytes(data)
        print('saved', IN / name, f'{len(data)/1e6:.2f} MB')

def missing_files():
    return [f for f in NEEDED_DATA if not (IN / f).exists()]

# Files uploaded with the Files panel land in /content (not in IN): move them into IN automatically
if IN_COLAB:
    for p in Path('/content').iterdir():
        if p.is_file() and (p.name in NEEDED_DATA or p.suffix.lower() in ('.tif', '.tiff') or 'admin_0_countries' in p.name.lower()):
            shutil.move(str(p), str(IN / p.name)); print('moved', p.name, '->', IN)

if missing_files() and IN_COLAB:
    print('Missing:', missing_files())
    print('Upload them now (also the 3 GeoTIFFs). For big TIFFs you can instead mount Drive and copy into', IN)
    upload_here()
print('Still missing:', missing_files())

# JSON writer: no NaN/Inf allowed, numpy types converted
def _np(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    raise TypeError(type(o))

def dump_json(obj, path, pretty=False):
    s = json.dumps(obj, ensure_ascii=False, allow_nan=False, default=_np,
                   indent=1 if pretty else None, separators=None if pretty else (',', ':'))
    Path(path).write_text(s, encoding='utf-8')
    return Path(path).stat().st_size

def round_coords(c, nd=5):
    return round(c, nd) if isinstance(c, float) else [round_coords(x, nd) for x in c]

print('IN =', IN.resolve(), '| OUT =', OUT.resolve())

# Texts shared by several cells (edit COST_NOTE_* to match your real cost model; I have not seen it)
DISCLAIMER_EN = ('Wind speeds come from the Global Wind Atlas, which is not validated for Azerbaijan here. '
                 'Capacity factors are gross (no wake, availability or electrical losses). Indicative only.')
DISCLAIMER_AZ = ('Külək sürətləri Qlobal Külək Atlasından götürülüb və Azərbaycan üçün burada doğrulanmayıb. '
                 'Tutum əmsalları brutto qiymətdir (izləmə, əlçatanlıq və elektrik itkiləri nəzərə alınmayıb). Yalnız istiqamətverici.')
COST_NOTE_EN = ('Cost is in relative model units (not currency) from the optimiser cost model; it is an assumption-based '
                'proxy, not a real project cost. Model: 1.0 unit per turbine, tower surcharge 0 / 10 / 25 % for 50 / 100 / 150 m, '
                'road cost 0.02 units per km to the nearest access point.')
COST_NOTE_AZ = ('Xərc nisbi model vahidlərindədir (valyuta deyil) və optimallaşdırıcının xərc modelindən gəlir; '
                'fərziyyəyə əsaslanan göstəricidir, real layihə xərci deyil. Model: hər turbin 1,0 vahid, qüllə əlavəsi 50 / 100 / 150 m üçün '
                '0 / 10 / 25 %, yol xərci ən yaxın çıxış nöqtəsinə hər km üçün 0,02 vahid.')
CONNECTION_NOTE = ('Indicative connection = straight line from each turbine to its nearest access point. '
                   'It is NOT a real cable or road design.')


## 2. Inspect the GWA GeoTIFFs (paste output back)

In [ ]:
# Cell 2: inspect the GWA GeoTIFFs (PASTE THIS OUTPUT BACK)
import rasterio
from rasterio.warp import transform as rio_transform

cand = pd.read_csv(IN / 'candidate_sites.csv')

# Map height -> file. Auto-detect 50/100/150 in the file name; override here if needed.
TIF_OVERRIDE = {}   # e.g. {50: 'x_50m.tif', 100: 'x_100m.tif', 150: 'x_150m.tif'}
HEIGHT_RE = re.compile(r'(?<!\d)(50|100|150)(?!\d)')
tifs = sorted(p for p in IN.iterdir() if p.suffix.lower() in ('.tif', '.tiff'))
print('GeoTIFFs found:', [p.name for p in tifs])
TIF = {int(h): IN / n for h, n in TIF_OVERRIDE.items()}
if not TIF:
    for p in tifs:
        hits = HEIGHT_RE.findall(p.stem)
        if len(hits) == 1:
            TIF[int(hits[0])] = p
assert sorted(TIF) == [50, 100, 150], f'Could not map heights from names, set TIF_OVERRIDE. Got {TIF}'
print('height -> file:', {h: p.name for h, p in sorted(TIF.items())})

SRC_INFO = {}
for h, p in sorted(TIF.items()):
    with rasterio.open(p) as ds:
        arr = ds.read(1, masked=True)
        valid = arr.compressed().astype('float64')
        info = dict(
            file=p.name, size_mb=round(p.stat().st_size / 1e6, 2), crs=str(ds.crs), width=ds.width, height=ds.height,
            count=ds.count, dtype=ds.dtypes[0], nodata=ds.nodata, res=ds.res, bounds=tuple(ds.bounds),
            scales=ds.scales[0], offsets=ds.offsets[0], units=ds.units[0], descr=ds.descriptions[0],
            tags=ds.tags(), band_tags=ds.tags(1))
        print(f'\n=== {h} m: {p.name} ===')
        for k, v in info.items():
            print(f'  {k:10s}: {v}')
        if ds.crs and ds.crs.is_geographic:
            lat0 = (ds.bounds.top + ds.bounds.bottom) / 2
            print(f'  approx cell size: {ds.res[0]*111320*math.cos(math.radians(lat0)):.0f} m x {ds.res[1]*110574:.0f} m')
        print(f'  masked (nodata) cells: {arr.mask.sum() if np.ndim(arr.mask) else 0} of {arr.size}'
              f' ({100*(arr.mask.sum() if np.ndim(arr.mask) else 0)/arr.size:.1f} %)')
        print(f'  valid min/mean/max: {valid.min():.3f} / {valid.mean():.3f} / {valid.max():.3f} (m/s expected)')
        print('  percentiles 1/50/99:', np.round(np.percentile(valid, [1, 50, 99]), 3))
        sus = int(((valid < -50) | (valid > 100)).sum())
        print(f'  suspicious values (<-50 or >100) that are NOT masked: {sus}')
        # Cross-check against candidate_sites.csv at the 251 points (verifies CRS and height mapping)
        xs, ys = rio_transform('EPSG:4326', ds.crs, cand.lon.tolist(), cand.lat.tolist())
        samp = np.array([float(np.ma.filled(v[0], np.nan)) for v in ds.sample(zip(xs, ys), masked=True)])
        d = samp - cand[f'ws{h}'].values
        ok = np.isfinite(d)
        print(f'  vs candidate_sites ws{h}: n={ok.sum()}  median|diff|={np.nanmedian(np.abs(d)):.3f}  '
              f'max|diff|={np.nanmax(np.abs(d)):.3f}  corr={np.corrcoef(samp[ok], cand[f"ws{h}"].values[ok])[0,1]:.4f}')
        info['bounds'] = list(info['bounds']); info['res'] = list(info['res'])
        if info['nodata'] is not None and not np.isfinite(info['nodata']): info['nodata'] = 'nan'
        SRC_INFO[h] = info
print('\nDone. If |diff| is large or corr is low, the CRS/height mapping or the candidate source differs: tell me.')


## 3. Azerbaijan outline (Natural Earth) + check

In [ ]:
# Cell 3: Azerbaijan outline from Natural Earth admin-0 (+ visual/numeric check, no silent edits)
import geopandas as gpd
import shapely
from shapely.geometry import shape, mapping, Point
from shapely.ops import unary_union
import matplotlib.pyplot as plt

NE_URLS = [
    'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip',
    'https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip',
]

def find_ne():
    for p in IN.iterdir():
        if 'admin_0_countries' in p.name.lower() and p.suffix.lower() in ('.zip', '.geojson', '.json', '.shp'):
            return p
    return None

ne_path = find_ne()
if ne_path is None:
    import requests
    for url in NE_URLS:
        try:
            r = requests.get(url, timeout=60); r.raise_for_status()
            ne_path = IN / Path(url).name; ne_path.write_bytes(r.content)
            print('downloaded', url, f'{len(r.content)/1e6:.1f} MB'); break
        except Exception as e:
            print('download failed:', url, '->', type(e).__name__, e)
if ne_path is None and IN_COLAB:
    print('Manual fallback: upload ne_10m_admin_0_countries.zip (naturalearthdata.com -> Downloads -> Admin 0 Countries)')
    upload_here(); ne_path = find_ne()
assert ne_path is not None, 'Put ne_10m_admin_0_countries.zip into the IN folder and re-run.'

world = gpd.read_file(f'zip://{ne_path}' if ne_path.suffix.lower() == '.zip' else ne_path)
world.columns = [c.upper() if c != 'geometry' else c for c in world.columns]
sel = world[(world.get('ADM0_A3') == 'AZE') | (world.get('ADMIN') == 'Azerbaijan')]
assert len(sel) == 1, f'expected 1 Azerbaijan row, got {len(sel)}'
print('Natural Earth row: ADMIN=%s ADM0_A3=%s' % (sel.iloc[0].get('ADMIN'), sel.iloc[0].get('ADM0_A3')))
sel = sel.set_crs(4326, allow_override=True) if sel.crs is None else sel.to_crs(4326)

OUTLINE_FULL = unary_union(sel.geometry.values)          # used for masking (full detail)
parts = list(OUTLINE_FULL.geoms) if OUTLINE_FULL.geom_type == 'MultiPolygon' else [OUTLINE_FULL]
areas = gpd.GeoSeries(parts, crs=4326).to_crs(6933).area / 1e6
print(f'polygons: {len(parts)}   total area: {areas.sum():,.0f} km2   bounds (lon/lat): {np.round(OUTLINE_FULL.bounds, 3)}')
print('largest parts (km2):', np.round(sorted(areas, reverse=True)[:6], 0))
print('For reference (my memory, approximate): Azerbaijan within internationally recognised borders is ~86,600 km2, '
      'Nakhchivan ~5,500 km2.')

# Sanity points (approximate coordinates from memory): are the key regions inside the outline?
CHECK_PTS = [  # name, lon, lat
    ('Baku', 49.87, 40.41), ('Nakhchivan city', 45.41, 39.21), ('Khankendi', 46.75, 39.82),
    ('Shusha', 46.75, 39.76), ('Lachin', 46.55, 39.64), ('Kalbajar', 46.04, 40.10),
    ('Zangilan', 46.65, 39.09), ('Fuzuli', 47.14, 39.60), ('Aghdam', 46.93, 39.99),
    ('Yerevan (should be OUTSIDE)', 44.51, 40.18), ('Tbilisi (should be OUTSIDE)', 44.79, 41.72)]
rows = [(n, OUTLINE_FULL.contains(Point(x, y))) for n, x, y in CHECK_PTS]
print(pd.DataFrame(rows, columns=['point', 'inside_outline']).to_string(index=False))

# Simplified copy for the site (~500 m tolerance), 4 decimals
OUTLINE_SIMPLE = OUTLINE_FULL.simplify(0.005, preserve_topology=True)
feat = {'type': 'FeatureCollection', 'features': [{
    'type': 'Feature',
    'properties': {'name': 'Azerbaijan', 'source': 'Natural Earth admin-0 countries (10m), simplified 0.005 deg'},
    'geometry': {'type': OUTLINE_SIMPLE.geom_type, 'coordinates': round_coords(mapping(OUTLINE_SIMPLE)['coordinates'], 4)}}]}
sz = dump_json(feat, OUT / 'azerbaijan_outline.geojson')
nv = lambda g: sum(len(p.exterior.coords) for p in (g.geoms if g.geom_type == 'MultiPolygon' else [g]))
print(f'azerbaijan_outline.geojson: {sz/1e3:.0f} KB, vertices {nv(OUTLINE_FULL)} -> {nv(OUTLINE_SIMPLE)}')

# Visual check
fig, ax = plt.subplots(figsize=(9, 6))
sel.plot(ax=ax, facecolor='#cfe3f7', edgecolor='k', lw=0.8)
for n, x, y in CHECK_PTS:
    ax.plot(x, y, 'r.' if 'OUTSIDE' not in n else 'b.'); ax.annotate(n.split(' (')[0], (x, y), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_title('Natural Earth outline used for masking (check Nakhchivan + Karabakh/East Zangezur)')
ax.set_aspect(1 / math.cos(math.radians(40.3))); plt.tight_layout()
plt.savefig(SITE / 'outline_check.png', dpi=110); plt.show()


## 4. Reproject, mask, colour, PNGs and click grids

In [ ]:
# Cell 4: reproject to EPSG:3857, mask, colour -> PNG (indexed), 1 km click grids (uint8)
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
import shapely
from pyproj import Transformer
from PIL import Image

PNG_RES_M = 350        # metres per pixel in EPSG:3857 (smaller = sharper, bigger files)
GRID_DEG  = 0.01       # click-grid cell size in degrees (~0.85 km x 1.1 km)
VMAX      = 10.0       # colour ramp saturates here ("10+ m/s")
NODATA_F  = -9999.0    # internal float sentinel
PNG_NODATA_IDX = 255   # transparent palette index

# GWA-like ramp: blue -> cyan -> green -> yellow -> orange -> red -> purple
RAMP = [(0, '#2c2f8f'), (1, '#2f56c4'), (2, '#2f8fd8'), (3, '#2fc4dc'), (4, '#35d0a8'), (5, '#52c95a'),
        (6, '#a4d63c'), (7, '#f0e03a'), (8, '#f5a52c'), (9, '#e8452a'), (10, '#8e24aa')]
hex2rgb = lambda s: tuple(int(s[i:i + 2], 16) for i in (1, 3, 5))
NLEV = 254   # colour levels 0..253, index 255 = transparent

def build_palette():
    xs = [v for v, _ in RAMP]; cols = np.array([hex2rgb(c) for _, c in RAMP], float)
    vals = np.linspace(0, VMAX, NLEV)
    pal = np.zeros((256, 3), np.uint8)
    for k in range(3):
        pal[:NLEV, k] = np.interp(vals, xs, cols[:, k]).round().astype(np.uint8)
    return pal

def load_src(h):
    with rasterio.open(TIF[h]) as ds:
        a = ds.read(1, masked=True).astype('float32')
        sc, off = ds.scales[0], ds.offsets[0]
        if sc not in (None, 1.0) or off not in (None, 0.0):
            a = a * (sc or 1.0) + (off or 0.0); print(f'  applied scale/offset {sc}/{off}')
        return a.filled(NODATA_F), ds.transform, ds.crs

# --- output geometry -------------------------------------------------------
t_to3857 = Transformer.from_crs(4326, 3857, always_xy=True)
t_to4326 = Transformer.from_crs(3857, 4326, always_xy=True)
minx, miny, maxx, maxy = OUTLINE_FULL.bounds
x0, y0 = t_to3857.transform(minx, miny); x1, y1 = t_to3857.transform(maxx, maxy)
left, bottom = math.floor(x0 / PNG_RES_M) * PNG_RES_M, math.floor(y0 / PNG_RES_M) * PNG_RES_M
right, top = math.ceil(x1 / PNG_RES_M) * PNG_RES_M, math.ceil(y1 / PNG_RES_M) * PNG_RES_M
W, H = int(round((right - left) / PNG_RES_M)), int(round((top - bottom) / PNG_RES_M))
png_tf = from_origin(left, top, PNG_RES_M, PNG_RES_M)
(west, south), (east, north) = t_to4326.transform(left, bottom), t_to4326.transform(right, top)
print(f'PNG: {W} x {H} px, EPSG:3857 bounds L{left:.0f} B{bottom:.0f} R{right:.0f} T{top:.0f}')
print(f'     lon/lat bounds W{west:.4f} S{south:.4f} E{east:.4f} N{north:.4f}')

outline_3857 = shapely.transform(OUTLINE_FULL, lambda a: np.column_stack(t_to3857.transform(a[:, 0], a[:, 1])))
mask_png = rasterize([(outline_3857, 1)], out_shape=(H, W), transform=png_tf, fill=0, dtype='uint8').astype(bool)

# click grid geometry (EPSG:4326, regular)
g_west, g_east = math.floor(minx / GRID_DEG) * GRID_DEG, math.ceil(maxx / GRID_DEG) * GRID_DEG
g_south, g_north = math.floor(miny / GRID_DEG) * GRID_DEG, math.ceil(maxy / GRID_DEG) * GRID_DEG
g_west, g_east, g_south, g_north = (round(v, 6) for v in (g_west, g_east, g_south, g_north))
NC, NR = int(round((g_east - g_west) / GRID_DEG)), int(round((g_north - g_south) / GRID_DEG))
grid_tf = from_origin(g_west, g_north, GRID_DEG, GRID_DEG)
mask_grid = rasterize([(OUTLINE_FULL, 1)], out_shape=(NR, NC), transform=grid_tf, fill=0, dtype='uint8').astype(bool)
print(f'Grid: {NC} cols x {NR} rows ({NC*NR/1e3:.0f} k cells)')

# --- per-height processing -------------------------------------------------
pal = build_palette()
layers, grids_f = {}, {}
for h in (50, 100, 150):
    print(f'\n{h} m')
    src, src_tf, src_crs = load_src(h)
    # PNG: bilinear into the 3857 grid
    dst = np.full((H, W), NODATA_F, np.float32)
    reproject(src, dst, src_transform=src_tf, src_crs=src_crs, dst_transform=png_tf, dst_crs='EPSG:3857',
              src_nodata=NODATA_F, dst_nodata=NODATA_F, resampling=Resampling.bilinear)
    has = dst > -9990
    miss = mask_png & ~has
    print(f'  cells inside outline but without raster data: {miss.sum()} ({100*miss.sum()/mask_png.sum():.2f} % of outline)')
    ok = mask_png & has
    idx = np.full((H, W), PNG_NODATA_IDX, np.uint8)
    idx[ok] = np.clip(np.round(dst[ok] / VMAX * (NLEV - 1)), 0, NLEV - 1).astype(np.uint8)
    im = Image.fromarray(idx); im.putpalette(pal.flatten().tolist())   # L -> P (the mode= argument of fromarray is removed in newer Pillow)
    png_path = OUT / f'wind_{h}m.png'
    im.save(png_path, optimize=True, transparency=PNG_NODATA_IDX)
    # grid: average into the 4326 grid
    gd = np.full((NR, NC), NODATA_F, np.float32)
    reproject(src, gd, src_transform=src_tf, src_crs=src_crs, dst_transform=grid_tf, dst_crs='EPSG:4326',
              src_nodata=NODATA_F, dst_nodata=NODATA_F, resampling=Resampling.average)
    gd[~(mask_grid & (gd > -9990))] = np.nan
    grids_f[h] = gd
    v = dst[ok]
    layers[h] = dict(png=png_path.name, grid=f'wind_grid_{h}m.bin',
                     stats_png_pixels=dict(min=round(float(v.min()), 3), mean=round(float(v.mean()), 3),
                                           max=round(float(v.max()), 3), share_above_vmax=round(float((v > VMAX).mean()), 4)))
    print(f'  PNG {png_path.stat().st_size/1e6:.2f} MB  value range in outline {v.min():.2f}..{v.max():.2f} m/s, '
          f'{100*(v > VMAX).mean():.2f} % above {VMAX}')

# shared scale for uint8 grids: q = round(v * SCALE), 255 = nodata
gmax = float(np.nanmax([np.nanmax(g) for g in grids_f.values()]))
SCALE = 20 if gmax * 20 <= 254 else int(254 // gmax)
print(f'\nGrid max {gmax:.2f} m/s -> SCALE={SCALE} (step {1/SCALE:.3f} m/s, max {254/SCALE:.1f} m/s)')
for h, g in grids_f.items():
    q = np.where(np.isnan(g), 255, np.clip(np.round(np.nan_to_num(g) * SCALE), 0, 254)).astype(np.uint8)
    (OUT / layers[h]['grid']).write_bytes(q.tobytes())
    layers[h]['grid_bytes'] = q.size
    print(f'  {layers[h]["grid"]}: {q.size/1e3:.0f} KB, valid cells {(q != 255).sum()}')

WIND_META = {
    'version': 1,
    'generated_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'heights_m': [50, 100, 150], 'unit': 'm/s',
    'layers': {str(h): layers[h] for h in layers},
    'png': {
        'crs': 'EPSG:3857', 'width': W, 'height': H, 'pixel_size_m': PNG_RES_M,
        'format': 'indexed PNG (palette), palette index 255 = fully transparent (outside Azerbaijan / no data)',
        'bounds_3857': dict(left=left, bottom=bottom, right=right, top=top),
        'bounds_lonlat': dict(west=west, south=south, east=east, north=north),
        'leaflet_bounds': [[south, west], [north, east]],
        'note': 'Image is gridded in EPSG:3857, so L.imageOverlay(url, leaflet_bounds) is geometrically exact.'},
    'ramp': {'stops': [{'v': v, 'color': c} for v, c in RAMP], 'vmin': 0, 'vmax': VMAX,
             'interpolation': 'linear in RGB, 254 levels', 'above_vmax': 'clamped to the last colour (10+ m/s)'},
    'grid': {
        'crs': 'EPSG:4326', 'dtype': 'uint8', 'ncols': NC, 'nrows': NR, 'west': g_west, 'north': g_north,
        'dlon': GRID_DEG, 'dlat': GRID_DEG, 'nodata': 255, 'scale': SCALE, 'offset': 0,
        'decode': 'value_ms = q / scale + offset; q == 255 means no data (outside outline)',
        'layout': 'raw row-major, rows north to south, columns west to east, no header',
        'lookup': 'col = floor((lon - west) / dlon); row = floor((north - lat) / dlat); byte index = row * ncols + col',
        'resampling': 'area average of the source raster; cells are kept only if their centre is inside the Natural Earth outline'},
    'source_rasters': {str(h): {k: SRC_INFO[h][k] for k in ('file', 'crs', 'width', 'height', 'res', 'nodata', 'dtype', 'bounds')}
                       for h in SRC_INFO},
    'attribution': [
        'Wind data: Global Wind Atlas (DTU / World Bank ESMAP)',
        'Country outline: Natural Earth',
        'Map tiles: Esri',
        'Map data: (c) OpenStreetMap contributors'],
    'licence_note': 'Data licences are not asserted here; check the Global Wind Atlas terms before publishing.',
    'disclaimer_en': DISCLAIMER_EN,
    'disclaimer_az': DISCLAIMER_AZ,
}

def write_meta(extra=None):
    m = dict(WIND_META); m.update(extra or {})
    return dump_json(m, OUT / 'wind_meta.json', pretty=True)
print('\nwind_meta.json:', write_meta(), 'bytes')


## 5. Vector files

In [ ]:
# Cell 5: vector files (region polygon, access points, candidate sites)
import geopandas as gpd
from shapely.geometry import shape, Point, mapping

best = pd.read_csv(IN / 'best_compromise_10x.csv')
par  = pd.read_csv(IN / 'pareto_solutions_10x.csv')
cand = pd.read_csv(IN / 'candidate_sites.csv')
acc  = pd.read_csv(IN / 'accessibility_points.csv')

# Access point descriptions come from YOUR OpenStreetMap notes (the CSV has no name column). Not verified here.
ACCESS_INFO = {
    'SUB01': dict(name='Qobustan Babacan', voltage='500/330/220 kV',
                  desc_en='substation', desc_az='yarımstansiyası'),
    'SUB02': dict(name=None, voltage='330/220/110 kV',
                  desc_en='substation in the east of the region', desc_az='regionun şərqində yarımstansiya'),
    'SUB04': dict(name='Yasma', voltage='330/220/110 kV',
                  desc_en='substation', desc_az='yarımstansiyası'),
    'SUB07': dict(name='Sengecal/Umbaki', voltage='220/110 kV',
                  desc_en='substation', desc_az='yarımstansiyası'),
    'JCT10': dict(name=None, voltage=None,
                  desc_en='trunk road junction (M1 / R6)', desc_az='magistral yol qovşağı (M1 / R6)'),
    'JCT20': dict(name=None, voltage=None,
                  desc_en='primary road junction (R6)', desc_az='əsas yol qovşağı (R6)'),
    'JCT06': dict(name=None, voltage=None,
                  desc_en='trunk junction (M4)', desc_az='magistral qovşaq (M4)'),
}
assert set(acc.id) == set(ACCESS_INFO), (set(acc.id) ^ set(ACCESS_INFO))

# --- region polygon ---------------------------------------------------------
reg = json.loads((IN / 'custom.geojson').read_text())
reg_geom = shape(reg['geometry'] if reg['type'] == 'Feature' else reg['features'][0]['geometry'])
reg_area = float(gpd.GeoSeries([reg_geom], crs=4326).to_crs(6933).area.iloc[0] / 1e6)
region_fc = {'type': 'FeatureCollection', 'features': [{
    'type': 'Feature',
    'properties': {'name': 'Study region', 'area_km2': round(reg_area, 1)},
    'geometry': {'type': 'Polygon', 'coordinates': round_coords(mapping(reg_geom)['coordinates'], 6)}}]}
print('region_polygon.geojson', dump_json(region_fc, OUT / 'region_polygon.geojson'), 'bytes; area (EPSG:6933) =', round(reg_area, 1), 'km2')

# --- access points ----------------------------------------------------------
feats = []
for _, r in acc.iterrows():
    i = ACCESS_INFO[r.id]
    feats.append({'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [round(r.lon, 6), round(r.lat, 6)]},
                  'properties': {'id': r.id, 'type': r.type, 'name': i['name'], 'voltage': i['voltage'],
                                 'description_en': i['desc_en'], 'description_az': i['desc_az'],
                                 'source': 'coordinates: accessibility_points.csv; names/voltage: user notes from OpenStreetMap (not verified)'}})
print('access_points.geojson', dump_json({'type': 'FeatureCollection', 'features': feats}, OUT / 'access_points.geojson'), 'bytes')

# --- candidate sites --------------------------------------------------------
feats = []
for _, r in cand.iterrows():
    feats.append({'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [round(r.lon, 6), round(r.lat, 6)]},
                  'properties': {'site_id': r.site_id, 'ws50': round(r.ws50, 3), 'ws100': round(r.ws100, 3), 'ws150': round(r.ws150, 3),
                                 'dist_km': round(r.dist_km, 3), 'nearest_access_id': r.nearest_access_id,
                                 'access_score': round(r.access_score, 4)}})
print('candidate_sites.geojson', dump_json({'type': 'FeatureCollection', 'features': feats}, OUT / 'candidate_sites.geojson'), 'bytes; n =', len(feats))

# quick numeric checks that need no raster
inside = [reg_geom.contains(Point(x, y)) for x, y in zip(cand.lon, cand.lat)]
print(f'candidate sites inside the region polygon: {sum(inside)} of {len(inside)}')


## 6. solutions.json and turbines.json

In [ ]:
# Cell 6: solutions.json and turbines.json (why_az / why_en built ONLY from numbers in the files)
STRICT = os.environ.get('STRICT', '1') == '1'   # True: a missing (site, tower) row in the table is an error

# ---- 1) solutions.json -----------------------------------------------------
def parse_genes(g):
    out = []
    for part in g.split(';'):
        s, t = part.split(':'); out.append((s.strip(), int(t)))
    return out

par = par.copy()
par['turb'] = par.genes.map(parse_genes)
best_idx = par.SVF.idxmax()
assert (par.SVF == par.SVF.max()).sum() == 1, 'tie for best SVF'
best_pairs = set(zip(best.site_id, best.tower_m.astype(int)))
print('best-SVF solution id:', int(par.loc[best_idx, 'id']), '| equals best_compromise_10x.csv:',
      set(par.loc[best_idx, 'turb']) == best_pairs)

solutions = []
for rank, (i, r) in enumerate(par.sort_values('SVF', ascending=False).iterrows(), start=1):
    solutions.append({
        'id': int(r.id), 'svf': round(float(r.SVF), 5), 'svf_rank': rank, 'is_best_svf': bool(i == best_idx),
        'feasible': bool(r.feasible),
        'objectives': {'aep_mwh': round(float(r.f1_aep_mwh), 1), 'aep_gwh': round(float(r.f1_aep_mwh) / 1000, 3),
                       'mean_access': round(float(r.f2_mean_access), 4), 'cost_units': round(float(r.f3_cost_units), 3)},
        'turbines': [{'site_id': s, 'tower_m': t} for s, t in r.turb]})
sol_doc = {
    'meta': {
        'n_solutions': len(solutions), 'best_svf_id': int(par.loc[best_idx, 'id']),
        'objectives': [
            {'key': 'aep_gwh', 'label_en': 'Annual energy (AEP, gross)', 'label_az': 'İllik enerji (AEP, brutto)', 'unit': 'GWh/yr', 'also': 'aep_mwh (MWh/yr)'},
            {'key': 'mean_access', 'label_en': 'Mean access score', 'label_az': 'Orta əlçatanlıq göstəricisi', 'unit': '0-1'},
            {'key': 'cost_units', 'label_en': 'Relative cost', 'label_az': 'Nisbi xərc', 'unit': 'model units'}],
        'direction_note': 'Directions inferred from the front (higher AEP goes with lower access and higher cost): '
                          'AEP and access are maximised, cost is minimised. Please confirm.',
        'svf_note': 'SVF value and ranking are taken as given from pareto_solutions_10x.csv (not recomputed here).',
        'order': 'solutions are sorted by SVF, highest first',
        'cost_note_en': COST_NOTE_EN, 'cost_note_az': COST_NOTE_AZ},
    'sites': {r.site_id: [round(r.lon, 6), round(r.lat, 6)] for r in cand.itertuples()},
    'solutions': solutions}
print('solutions.json', dump_json(sol_doc, OUT / 'solutions.json'), 'bytes')

# ---- 2) per-(site, tower) table -------------------------------------------
tbl = pd.read_csv(IN / 'site_turbine_table.csv')
print('site_turbine_table.csv columns:', list(tbl.columns), '| rows:', len(tbl))
COLMAP = {}   # set by hand if auto-detect fails, e.g. {'site': 'site_id', 'tower': 'tower_m', 'aep': 'aep_mwh', ...}
CANDS = {'site': ['site_id', 'site'], 'tower': ['tower_m', 'tower', 'tower_height_m', 'hub_height_m', 'hub_m'],
         'aep': ['aep_mwh', 'aep'], 'cost': ['cost_units', 'cost'],
         'hub_ws': ['hub_ws', 'ws_hub', 'hub_wind_speed'], 'cf': ['capacity_factor', 'cf']}
FUZZY = {'site': ['site'], 'tower': ['tower'], 'aep': ['aep'], 'cost': ['cost'], 'hub_ws': ['hub', 'ws'], 'cf': ['capacity']}
low = {c.lower(): c for c in tbl.columns}
cm = {}
for k, names in CANDS.items():
    if k in COLMAP: cm[k] = COLMAP[k]; continue
    hit = [low[n] for n in names if n in low] or [c for c in tbl.columns if all(w in c.lower() for w in FUZZY[k])]
    if k == 'cf' and not hit: cm[k] = None; continue
    if not hit or (len(hit) > 1 and not [n for n in names if n in low]):
        raise ValueError(f'Cannot map "{k}" uniquely, candidates {hit}. Set COLMAP by hand. Columns: {list(tbl.columns)}')
    cm[k] = hit[0]
print('column mapping:', cm)
tbl = tbl.rename(columns={v: k for k, v in cm.items() if v})
tbl['tower'] = tbl['tower'].astype(int)
if tbl.duplicated(['site', 'tower']).any():
    raise ValueError('site_turbine_table has several rows per (site, tower) - tell me which one to use '
                     f'(columns: {list(tbl.columns)}).')
T = tbl.set_index(['site', 'tower'])

# capacity factor fallback: only if the table has none; implied rated power from the best-compromise file
P_IMPLIED = (best.aep_mwh / (8760 * best.capacity_factor)).round(6)
if cm['cf'] is None:
    assert P_IMPLIED.nunique() == 1, f'implied rated power is not constant: {P_IMPLIED.tolist()}'
    P_MW = float(P_IMPLIED.iloc[0]); T['cf'] = T['aep'] / (8760 * P_MW); cf_source = f'derived: AEP / (8760 h x {P_MW} MW implied by best_compromise_10x.csv)'
else:
    cf_source = 'site_turbine_table.csv'
print('capacity factor source:', cf_source, '| implied rated power from best file (MW):', sorted(set(P_IMPLIED)))

# verify the table against best_compromise_10x.csv (validates the column mapping)
chk = []
for r in best.itertuples():
    row = T.loc[(r.site_id, int(r.tower_m))]
    chk.append(max(abs(row.aep - r.aep_mwh) / r.aep_mwh, abs(row.cost - r.cost_units) / r.cost_units,
                   abs(row.hub_ws - r.hub_ws) / r.hub_ws))
print(f'table vs best_compromise: max relative difference (aep, cost, hub_ws) = {max(chk):.2e}')

# ---- 3) text helpers -------------------------------------------------------
acc_by_id = acc.set_index('id'); cand_by_id = cand.set_index('site_id')
def az_num(x, nd=0):
    return f'{x:,.{nd}f}'.replace(',', ' ').replace('.', ',')
def label(aid, lang):
    i = ACCESS_INFO[aid]; nm = (i['name'] + ' ') if i['name'] else ''
    return f"{aid} – {nm}{i['desc_en'] if lang == 'en' else i['desc_az']}"
def percentile(h, ws):   # share of the 251 sites with hub-height ws <= this one (same definition as the file)
    return 100 * float((cand[f'ws{h}'] <= ws + 1e-9).sum()) / len(cand)

def alt_clause(t, a, lang):
    dA, dC = a['aep'] - t['aep'], a['cost'] - t['cost']; pA = 100 * dA / t['aep']
    if lang == 'en':
        e = 'gives the same energy' if abs(dA) < 0.5 else f"gives {abs(dA):,.0f} MWh/yr ({abs(pA):.1f} %) {'more' if dA > 0 else 'less'} energy"
        c = 'costs the same' if abs(dC) < 5e-4 else f"costs {abs(dC):.3f} units {'more' if dC > 0 else 'less'}"
        return f"a {a['tower']} m tower {e} and {c}"
    e = 'illik enerji eynidir' if abs(dA) < 0.5 else f"illik enerji {az_num(abs(dA))} MWh ({az_num(abs(pA), 1)} %) {'çox' if dA > 0 else 'az'}"
    c = 'xərc eynidir' if abs(dC) < 5e-4 else f"xərc {az_num(abs(dC), 3)} vahid {'çox' if dC > 0 else 'az'}"
    return f"{a['tower']} m qüllə olsaydı, {e}, {c} olardı"

def build_why(d, alts):
    if alts:
        en_tw = f"The tower is {d['tower']} m; at the same site, " + '; '.join(alt_clause(d, a, 'en') for a in alts) + '.'
        az_tw = f"Qüllə hündürlüyü {d['tower']} m-dir; eyni sahədə " + '; '.join(alt_clause(d, a, 'az') for a in alts) + '.'
    else:
        en_tw = f"The tower is {d['tower']} m; no alternative tower height is listed for this site."
        az_tw = f"Qüllə hündürlüyü {d['tower']} m-dir; bu sahə üçün cədvəldə alternativ qüllə hündürlüyü yoxdur."
    en = (f"Hub-height wind speed is {d['hub_ws']:.2f} m/s at {d['tower']} m, at percentile {d['hub_ws_percentile']:.1f} "
          f"among the 251 candidate sites at the same height. "
          f"The nearest access point is {label(d['nearest_access_id'], 'en')}, {d['dist_km']:.1f} km away in a straight line "
          f"(access score {d['access_score']:.2f}). "
          f"{en_tw} "
          f"Gross capacity factor is {d['capacity_factor']*100:.1f} % (no wake, availability or electrical losses). "
          f"This turbine is part of an NSGA-II compromise solution, not 'the' optimum. {COST_NOTE_EN}")
    az = (f"{d['tower']} m hündürlükdə küləyin sürəti {az_num(d['hub_ws'], 2)} m/s-dir; bu, eyni hündürlükdə 251 namizəd sahə "
          f"arasında {az_num(d['hub_ws_percentile'], 1)} persentil səviyyəsidir. "
          f"Ən yaxın çıxış nöqtəsi {label(d['nearest_access_id'], 'az')} olub, düz xətt üzrə {az_num(d['dist_km'], 1)} km məsafədədir "
          f"(əlçatanlıq göstəricisi {az_num(d['access_score'], 2)}). "
          f"{az_tw} "
          f"Brutto tutum əmsalı {az_num(d['capacity_factor']*100, 1)} %-dir (izləmə, əlçatanlıq və elektrik itkiləri nəzərə alınmayıb). "
          f"Bu turbin NSGA-II kompromis həllinin bir hissəsidir, 'yeganə' optimum deyil. {COST_NOTE_AZ}")
    return az, en

# ---- 4) turbines.json ------------------------------------------------------
pairs = sorted({p for s in solutions for p in [(t['site_id'], t['tower_m']) for t in s['turbines']]})
count_in = {p: sum(1 for s in solutions if any((t['site_id'], t['tower_m']) == p for t in s['turbines'])) for p in pairs}
HEIGHTS = [50, 100, 150]
turbines, missing = {}, []
for site, tw in pairs:
    if (site, tw) not in T.index: missing.append((site, tw)); continue
    row = T.loc[(site, tw)]; c = cand_by_id.loc[site]; a_id = c.nearest_access_id; ap = acc_by_id.loc[a_id]
    d = dict(site_id=site, tower_m=tw, lon=round(float(c.lon), 6), lat=round(float(c.lat), 6),
             hub_ws=round(float(row.hub_ws), 3), hub_ws_percentile=round(percentile(tw, float(row.hub_ws)), 1),
             aep_mwh=round(float(row.aep), 1), capacity_factor=round(float(row.cf), 4), cost_units=round(float(row.cost), 4),
             access_score=round(float(c.access_score), 4), dist_km=round(float(c.dist_km), 3),
             nearest_access_id=a_id, nearest_access_type=str(ap.type), nearest_access_name=ACCESS_INFO[a_id]['name'],
             nearest_access_voltage=ACCESS_INFO[a_id]['voltage'],
             ws50=round(float(c.ws50), 3), ws100=round(float(c.ws100), 3), ws150=round(float(c.ws150), 3))
    # neighbouring tower heights at the same site (50 -> 100; 100 -> 50 and 150; 150 -> 100)
    k = HEIGHTS.index(tw); nb = [HEIGHTS[j] for j in (k - 1, k + 1) if 0 <= j < len(HEIGHTS)]
    alts = []
    for h2 in nb:
        if (site, h2) in T.index:
            r2 = T.loc[(site, h2)]
            alts.append(dict(tower=h2, aep=float(r2.aep), cost=float(r2.cost)))
    t_self = dict(tower=tw, aep=float(row.aep), cost=float(row.cost))
    az_txt, en_txt = build_why(dict(d, tower=tw, aep=t_self['aep'], cost=t_self['cost']), alts)
    d['alternatives'] = [dict(tower_m=a['tower'], delta_aep_mwh=round(a['aep'] - t_self['aep'], 1),
                              delta_aep_pct=round(100 * (a['aep'] - t_self['aep']) / t_self['aep'], 2),
                              delta_cost_units=round(a['cost'] - t_self['cost'], 4)) for a in alts]
    d['in_n_solutions'] = count_in[(site, tw)]; d['in_best_svf_solution'] = (site, tw) in set(par.loc[best_idx, 'turb'])
    d['connection'] = {'kind': 'indicative straight line (not a cable design)',
                       'coordinates': [[d['lon'], d['lat']], [round(float(ap.lon), 6), round(float(ap.lat), 6)]],
                       'length_km_file': d['dist_km']}
    d['why_az'], d['why_en'] = az_txt, en_txt
    turbines[f'{site}_{tw}'] = d
if missing:
    msg = f'{len(missing)} (site, tower) pairs have no row in site_turbine_table.csv: {missing[:8]}...'
    if STRICT: raise KeyError(msg)
    print('WARNING (STRICT off):', msg)

turb_doc = {'meta': {'n_turbines': len(turbines), 'key_format': '<site_id>_<tower_m>', 'capacity_factor_source': cf_source,
                     'percentile_definition': 'share (%) of the 251 candidate sites whose wind speed at the same height is <= this turbine (same as best_compromise_10x.csv)',
                     'alternatives_definition': 'deltas = (neighbouring tower height) minus (this tower) at the same site, from site_turbine_table.csv',
                     'dist_km_note': 'dist_km is taken from candidate_sites.csv (straight-line distance to the nearest access point)',
                     'connection_note': CONNECTION_NOTE, 'cost_note_en': COST_NOTE_EN, 'cost_note_az': COST_NOTE_AZ,
                     'disclaimer_en': DISCLAIMER_EN, 'disclaimer_az': DISCLAIMER_AZ},
            'turbines': turbines}
print('turbines.json', dump_json(turb_doc, OUT / 'turbines.json'), 'bytes;', len(turbines), 'entries for', len(pairs), 'pairs')
k0 = f"{best.site_id.iloc[0]}_{int(best.tower_m.iloc[0])}"
if k0 in turbines:
    print('\nSample', k0, '\nwhy_en:', turbines[k0]['why_en'], '\nwhy_az:', turbines[k0]['why_az'])

if 'WIND_META' in globals():   # final metadata (adds notes that belong to the vector files)
    print('wind_meta.json', write_meta({'connection_note': CONNECTION_NOTE, 'cost_note_en': COST_NOTE_EN, 'cost_note_az': COST_NOTE_AZ}), 'bytes')
else:
    print('wind_meta.json not updated (run cell 4 first).')


## 7. Self-checks and alignment plot

In [ ]:
# Cell 7: self-checks (PASS / FAIL / SKIP) + alignment plot
import matplotlib.pyplot as plt
from PIL import Image
from pyproj import Transformer
results = []
def check(name, ok, detail=''):
    results.append((name, ok)); print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")
def skip(name, why): print(f'[SKIP] {name} ({why})')

J = lambda n: json.loads((OUT / n).read_text(encoding='utf-8'))
sol, trb = J('solutions.json'), J('turbines.json')
HAVE_RASTER = (OUT / 'wind_meta.json').exists() and all((OUT / f'wind_{h}m.png').exists() for h in (50, 100, 150))

# 1) every turbine of every solution has an entry
need = {f"{t['site_id']}_{t['tower_m']}" for s in sol['solutions'] for t in s['turbines']}
check('every (site, tower) of every solution is in turbines.json', need <= set(trb['turbines']),
      f"({len(need)} needed, {len(need & set(trb['turbines']))} found)")
check('91 solutions, each with 10 distinct sites, all feasible',
      len(sol['solutions']) == 91 and all(len({t['site_id'] for t in s['turbines']}) == 10 and s['feasible'] for s in sol['solutions']))
check('exactly one best-SVF flag', sum(s['is_best_svf'] for s in sol['solutions']) == 1)
check('every turbine has why_az and why_en', all(t.get('why_az') and t.get('why_en') for t in trb['turbines'].values()))

# 2) no NaN / Inf anywhere (dump_json already refuses them; re-scan the files as text)
bad = [n for n in sorted(p.name for p in OUT.glob('*.json')) + sorted(p.name for p in OUT.glob('*.geojson'))
       if re.search(r'\bNaN\b|\bInfinity\b', (OUT / n).read_text(encoding='utf-8'))]
check('no NaN/Infinity in any JSON/GeoJSON', not bad, str(bad))

# 3) coordinates inside the PNG bounds
if HAVE_RASTER:
    wm = J('wind_meta.json'); b = wm['png']['bounds_lonlat']
    pts = [(v['lon'], v['lat']) for v in trb['turbines'].values()] + [tuple(v) for v in sol['sites'].values()]
    pts += [tuple(f['geometry']['coordinates']) for f in J('access_points.geojson')['features']]
    pts += [tuple(c) for c in J('region_polygon.geojson')['features'][0]['geometry']['coordinates'][0]]
    inb = [(b['west'] <= x <= b['east'] and b['south'] <= y <= b['north']) for x, y in pts]
    check('all turbine / site / access / region coordinates inside PNG lon/lat bounds', all(inb), f'({sum(inb)}/{len(inb)})')
else:
    skip('coordinates inside PNG bounds', 'raster cell 4 not run')

# 4) candidate sites / access ids
cs = J('candidate_sites.geojson'); ap = J('access_points.geojson')
check('candidate_sites.geojson has 251 points with all required fields',
      len(cs['features']) == 251 and all({'ws50', 'ws100', 'ws150', 'dist_km', 'nearest_access_id', 'access_score'} <= set(f['properties']) for f in cs['features']))
ids = {f['properties']['id'] for f in ap['features']}
check('nearest_access_id of every site exists in access_points.geojson', {f['properties']['nearest_access_id'] for f in cs['features']} <= ids)

# 5) wind layers: PNG, grid, alignment against candidate_sites
if HAVE_RASTER:
    g = wm['grid']
    for h in (50, 100, 150):
        im = Image.open(OUT / f'wind_{h}m.png')
        check(f'wind_{h}m.png size matches meta, palette mode, transparency present',
              im.size == (wm['png']['width'], wm['png']['height']) and im.mode == 'P' and 'transparency' in im.info, f'{im.size}, {im.mode}')
        raw = np.frombuffer((OUT / wm['layers'][str(h)]['grid']).read_bytes(), np.uint8)
        check(f'wind_grid_{h}m.bin length = ncols*nrows and < 1 MB', raw.size == g['ncols'] * g['nrows'] and raw.size < 1e6, f'({raw.size/1e3:.0f} KB)')
        # decode the grid at candidate sites and compare with ws{h} from the CSV (alignment test in numbers)
        cc = np.floor((cand.lon.values - g['west']) / g['dlon']).astype(int)
        rr = np.floor((g['north'] - cand.lat.values) / g['dlat']).astype(int)
        ok = (cc >= 0) & (cc < g['ncols']) & (rr >= 0) & (rr < g['nrows'])
        q = raw.reshape(g['nrows'], g['ncols'])[rr[ok], cc[ok]]
        v = np.where(q == 255, np.nan, q / g['scale'] + g['offset'])
        diff = np.abs(v - cand[f'ws{h}'].values[ok])
        print(f'       grid vs candidate ws{h}: valid {np.isfinite(v).sum()}/{ok.sum()}, median|diff|={np.nanmedian(diff):.3f}, '
              f'p95={np.nanpercentile(diff, 95):.3f}, max={np.nanmax(diff):.3f} m/s (1 km averaging explains small differences)')
        check(f'grid {h} m agrees with candidate_sites (median |diff| < 0.5 m/s: heuristic threshold)', np.nanmedian(diff) < 0.5)
else:
    skip('PNG / grid checks', 'raster cell 4 not run')

# 6) sizes
sizes = {p.name: p.stat().st_size for p in sorted(OUT.iterdir())}
for n, s in sizes.items(): print(f'   {n:28s} {s/1e3:10.1f} KB')
tot = sum(sizes.values()); check('total size of site/data under 20 MB', tot < 20e6, f'({tot/1e6:.2f} MB)')

# 7) alignment plot: PNG + 10 best turbines + access points + region polygon (+ outline)
if HAVE_RASTER:
    L, B, R, T_ = (wm['png']['bounds_3857'][k] for k in ('left', 'bottom', 'right', 'top'))
    to3857 = Transformer.from_crs(4326, 3857, always_xy=True)
    def xy(lons, lats): return to3857.transform(np.asarray(lons), np.asarray(lats))
    fig, axs = plt.subplots(1, 2, figsize=(15, 6))
    for ax, zoom in zip(axs, (False, True)):
        ax.imshow(Image.open(OUT / 'wind_100m.png').convert('RGBA'), extent=(L, R, B, T_), origin='upper', interpolation='nearest')
        rg = np.array(J('region_polygon.geojson')['features'][0]['geometry']['coordinates'][0]); rx, ry = xy(rg[:, 0], rg[:, 1])
        ax.plot(rx, ry, 'k-', lw=1.5, label='region polygon')
        o = J('azerbaijan_outline.geojson')['features'][0]['geometry']
        for poly in (o['coordinates'] if o['type'] == 'MultiPolygon' else [o['coordinates']]):
            ox, oy = xy(*np.array(poly[0]).T); ax.plot(ox, oy, color='w', lw=0.6)
        bx, by = xy(best.lon, best.lat); ax.scatter(bx, by, c='k', s=40 if zoom else 12, marker='^', label='10 best turbines', zorder=5)
        if zoom:
            for s_, x_, y_ in zip(best.site_id, bx, by): ax.annotate(s_, (x_, y_), fontsize=7, xytext=(3, 3), textcoords='offset points')
        ax_df = acc; px, py = xy(ax_df.lon, ax_df.lat); ax.scatter(px, py, c='magenta', s=50 if zoom else 15, marker='s', label='access points', zorder=5)
        if zoom:
            for i_, x_, y_ in zip(ax_df.id, px, py): ax.annotate(i_, (x_, y_), fontsize=8, color='magenta', xytext=(4, -9), textcoords='offset points')
            ax.set_xlim(rx.min() - 15000, rx.max() + 15000); ax.set_ylim(ry.min() - 15000, ry.max() + 15000)
        ax.set_title('wind_100m.png ' + ('(zoom on region)' if zoom else '(full extent)')); ax.set_aspect('equal'); ax.legend(loc='lower left', fontsize=7)
    plt.tight_layout(); plt.savefig(SITE / 'alignment_check.png', dpi=110); plt.show()
else:
    skip('alignment plot', 'raster cell 4 not run')

print(f"\nSUMMARY: {sum(ok for _, ok in results)} PASS, {sum(not ok for _, ok in results)} FAIL")
for n, ok in results:
    if not ok: print('  FAILED:', n)


## 8. Manifest, zip, download

In [ ]:
# Cell 8: manifest, zip, download
DESC = {
    'wind_50m.png': 'Mean wind speed at 50 m, EPSG:3857 indexed PNG overlay (transparent outside Azerbaijan)',
    'wind_100m.png': 'Mean wind speed at 100 m, EPSG:3857 indexed PNG overlay (transparent outside Azerbaijan)',
    'wind_150m.png': 'Mean wind speed at 150 m, EPSG:3857 indexed PNG overlay (transparent outside Azerbaijan)',
    'wind_grid_50m.bin': 'Click-query grid at 50 m: raw uint8, ~1 km, decode with wind_meta.json (q/scale, 255 = no data)',
    'wind_grid_100m.bin': 'Click-query grid at 100 m: raw uint8, ~1 km, decode with wind_meta.json (q/scale, 255 = no data)',
    'wind_grid_150m.bin': 'Click-query grid at 150 m: raw uint8, ~1 km, decode with wind_meta.json (q/scale, 255 = no data)',
    'wind_meta.json': 'Metadata: bounds (lon/lat and EPSG:3857), image/grid layout, colour ramp, attribution, disclaimer, notes',
    'azerbaijan_outline.geojson': 'Azerbaijan outline (Natural Earth admin-0, simplified)',
    'region_polygon.geojson': 'Study region polygon (about 2108 km2)',
    'access_points.geojson': 'Grid / road access points (id, type, name, voltage where known)',
    'candidate_sites.geojson': '251 candidate sites with ws50/ws100/ws150, dist_km, nearest_access_id, access_score',
    'solutions.json': '91 Pareto solutions (objectives, SVF, best-SVF flag, 10 turbines each) + site_id -> lon/lat lookup',
    'turbines.json': 'Every (site, tower) pair used in any solution: numbers, comparisons, connection line, why_az / why_en',
    'data_manifest.md': 'This list',
}
(OUT / 'data_manifest.md').write_text('# site/data manifest\n\n', encoding='utf-8')   # placeholder so it is listed
def build_manifest():
    lines = ['# site/data manifest', '']
    for p in sorted(OUT.iterdir()):
        kb = p.stat().st_size / 1024
        lines.append(f"- `{p.name}` ({kb:,.1f} KB): {DESC.get(p.name, 'NOT DESCRIBED')}")
    tot = sum(p.stat().st_size for p in OUT.iterdir()) / 1e6
    lines += ['', f'Total: {tot:.2f} MB', '', 'Attribution: Global Wind Atlas (DTU / World Bank ESMAP); Natural Earth; Esri and OpenStreetMap map tiles.']
    (OUT / 'data_manifest.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
build_manifest(); build_manifest()   # second pass so the manifest's own size is correct
print((OUT / 'data_manifest.md').read_text(encoding='utf-8'))

zip_path = SITE / 'data.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()): z.write(p, f'data/{p.name}')
print(f'{zip_path}: {zip_path.stat().st_size/1e6:.2f} MB')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
